# Analysis outline

See [README.md](README.md).

# Set up

Run the following cell.  It loads some required functions.

In [36]:
run kondrashov

# Search for human sequences

Below we try to retrieve human sequences for each gene programatically.  We
choose the [RefSeq Select](https://www.ncbi.nlm.nih.gov/refseq/refseq_select/) database.

In [ ]:
# genes in Kondrashov et al. (2002) are already listed in kondrashov.py
print(loci)

['ABCD1', 'ALPL', 'AR', 'ATP7B', 'BTK', 'CASR', 'CBS', 'CFTR', 'CYBB', 'F7', 'F8', 'F9', 'G6PD', 'GALT', 'GBA', 'GJB1', 'HBB', 'HPRT1', 'IL2RG', 'KCNH2', 'KCNQ1', 'L1CAM', 'LDLR', 'MPZ', 'MYH7', 'TYR', 'PAH', 'PMM2', 'RHO', 'TP53', 'TTR', 'VWF']


In [154]:
genes = {}
for locus in loci:
    query = f"Homo sapiens[Organism] AND {locus}[Gene Name] AND refseq_select[Filter]"
    handle = Entrez.esearch(db="nucleotide", term=query, retmax=100)
    record = Entrez.read(handle)
    handle.close()
    print(f"{locus}: {record['Count']} ->", record['IdList'][:4])
    genes.update({locus: record['IdList']})

ABCD1: 1 -> ['1519313321']
ALPL: 1 -> ['1519315936']
AR: 3 -> ['1654124212', '1519245710', '1519243240']
ATP7B: 1 -> ['1677498587']
BTK: 1 -> ['1780222528']
CASR: 1 -> ['1677537479']
CBS: 1 -> ['1780222567']
CFTR: 1 -> ['1732746288']
CYBB: 1 -> ['1732746192']
F7: 1 -> ['1779521762']
F8: 1 -> ['1812229661']
F9: 1 -> ['1732746198']
G6PD: 1 -> ['1497290737']
GALT: 1 -> ['1677502190']
GBA1: 1 -> ['1519244100']
GJB1: 1 -> ['1606717073']
HBB: 1 -> ['1401724401']
HPRT1: 1 -> ['1519243265']
IL2RG: 1 -> ['1780222514']
KCNH2: 1 -> ['1732746325']
KCNQ1: 1 -> ['1732746161']
L1CAM: 1 -> ['1653961419']
LDLR: 1 -> ['1732746181']
MPZ: 1 -> ['1519245315']
MYH7: 1 -> ['1519242569']
TYR: 1 -> ['1519243869']
PAH: 1 -> ['1519244862']
PMM2: 1 -> ['1519243247']
RHO: 2 -> ['1519316345', '169808383']
TP53: 1 -> ['1808862652']
TTR: 1 -> ['1780222569']
VWF: 1 -> ['1813372008']


Two of the genes, AR and RHO, have more than one RefSeq Select transcripts.
Let's look at them more closely.

* AR: only the first sequence is valid.

* RHO: only the second sequence is valid.

In [155]:
# AR
for i in genes['AR']:
    stream = Entrez.efetch(db="nucleotide", id=i, rettype="gb", retmode="text")
    record2 = SeqIO.read(stream, "genbank")
    stream.close()
    print(f"ID: {i}")
    print(f"{record2.id}: {record2.description}\nLength: {len(record2.seq)} nt\n")

ID: 1654124212
NM_000044.6: Homo sapiens androgen receptor (AR), transcript variant 1, mRNA
Length: 10667 nt

ID: 1519245710
NM_001657.4: Homo sapiens amphiregulin (AREG), mRNA
Length: 1234 nt

ID: 1519243240
NM_001628.4: Homo sapiens aldo-keto reductase family 1 member B (AKR1B1), transcript variant 1, mRNA
Length: 1361 nt



In [156]:
# remove wrong sequences
del genes['AR'][2]
del genes['AR'][1]
genes['AR']

['1654124212']

In [157]:
# RHO
for i in genes['RHO']:
    stream = Entrez.efetch(db="nucleotide", id=i, rettype="gb", retmode="text")
    record2 = SeqIO.read(stream, "genbank")
    stream.close()
    print(f"ID: {i}")
    print(f"{record2.id}: {record2.description}\nLength: {len(record2.seq)} nt\n")

ID: 1519316345
NM_014578.4: Homo sapiens ras homolog family member D (RHOD), transcript variant 1, mRNA
Length: 1104 nt

ID: 169808383
NM_000539.3: Homo sapiens rhodopsin (RHO), mRNA
Length: 2768 nt



In [101]:
# remove wrong sequence
del genes['RHO'][0]
genes['RHO']

['169808383']

In [102]:
for gene in genes:
    stream = Entrez.efetch(db="nucleotide", id=genes[gene][0], rettype="gb", retmode="text")
    record2 = SeqIO.read(stream, "genbank")
    stream.close()
    genes[gene].append(record2.id)
    print(gene, genes[gene])

ABCD1 ['1519313321', 'NM_000033.4']
ALPL ['1519315936', 'NM_000478.6']
AR ['1654124212', 'NM_000044.6']
ATP7B ['1677498587', 'NM_000053.4']
BTK ['1780222528', 'NM_000061.3']
CASR ['1677537479', 'NM_000388.4']
CBS ['1780222567', 'NM_000071.3']
CFTR ['1732746288', 'NM_000492.4']
CYBB ['1732746192', 'NM_000397.4']
F7 ['1779521762', 'NM_019616.4']
F8 ['1812229661', 'NM_000132.4']
F9 ['1732746198', 'NM_000133.4']
G6PD ['1497290737', 'NM_001360016.2']
GALT ['1677502190', 'NM_000155.4']
GBA ['1519244100', 'NM_000157.4']
GJB1 ['1606717073', 'NM_000166.6']
HBB ['1401724401', 'NM_000518.5']
HPRT1 ['1519243265', 'NM_000194.3']
IL2RG ['1780222514', 'NM_000206.3']
KCNH2 ['1732746325', 'NM_000238.4']
KCNQ1 ['1732746161', 'NM_000218.3']
L1CAM ['1653961419', 'NM_001278116.2']
LDLR ['1732746181', 'NM_000527.5']
MPZ ['1519245315', 'NM_000530.8']
MYH7 ['1519242569', 'NM_000257.4']
TYR ['1519243869', 'NM_000372.5']
PAH ['1519244862', 'NM_000277.3']
PMM2 ['1519243247', 'NM_000303.3']
RHO ['169808383', 'NM_

In [141]:
# pull protein ID
handle_link = Entrez.elink(dbfrom="nuccore", db="protein", id="NM_000552.5")
link_record = Entrez.read(handle_link)
protein_id = link_record[0]["LinkSetDb"][0]["Link"][0]["Id"]
protein_id

'1813372009'

In [ ]:
# pull protein sequence ID
stream = Entrez.efetch(db="protein", id='1813372009', rettype="gb", retmode="text")
record2 = SeqIO.read(stream, "genbank")
stream.close()
record2.id

'NP_000543.3'

# Search for orthologs

Primates are represented by [Registry Number
txid9443](https://www.ncbi.nlm.nih.gov/mesh/68011323).

Searching in the `Gene` database appears to retrieve the sequences that can be
obtained by doing the following:

* In the [NCBI query page](https://www.ncbi.nlm.nih.gov/labs/gquery/) search for
  `Homo sapiens {Gene Name}` (remove braces).  

* When the search results come up click on `Orthologs`.  

* Under `Selected taxa` write **Primates**. 

This was the procedure used in Hannah Esopenko's analysis.

In [91]:
orth = {}
for locus in loci:
    query = f"{locus}[Gene Name] AND txid9443[Organism]"
    handle = Entrez.esearch(db="gene", term=query, retmax=100)
    record = Entrez.read(handle)
    handle.close()
    print(f"{locus}: {record['Count']} ->", record['IdList'][:4])
    orth.update({locus: record['IdList']})

ABCD1: 35 -> ['215', '6367', '696794', '465923']
ALPL: 35 -> ['249', '717809', '456600', '100401643']
AR: 39 -> ['367', '374', '231', '574293']
ATP7B: 34 -> ['540', '713781', '452734', '101866907']
BTK: 34 -> ['695', '703000', '465759', '102123534']
CASR: 35 -> ['846', '714441', '460628', '101867346']
CBS: 32 -> ['875', '736626', '100423684', '100414619']
CFTR: 37 -> ['1080', '574346', '463674', '100137035']
CYBB: 34 -> ['1536', '696542', '465566', '102122418']
F7: 32 -> ['2155', '721933', '744785', '100410381']
F8: 33 -> ['2157', '100424151', '473838', '100393161']
F9: 34 -> ['2158', '695578', '465887', '102146175']
G6PD: 33 -> ['2539', '701137', '743041', '100388810']
GALT: 38 -> ['2592', '145173', '473219', '106993709']
GBA: 28 -> ['2629', '449571', '719103', '123634063']
GJB1: 35 -> ['2705', '706327', '465696', '102128042']
HBB: 25 -> ['3043', '450978', '101926697', '715559']
HPRT1: 34 -> ['3251', '735894', '101867079', '709186']
IL2RG: 36 -> ['3561', '641338', '101059843', '102144

In [ ]:
# MPZ: 35 -> ['4359', '719969', '747431', '102115938']
handle = Entrez.efetch(db="gene", id='4359', rettype="xml", retmode="text")
record = x2d.parse(handle.read().decode('utf-8'))
handle.close()
record

{'Entrezgene-Set': {'Entrezgene': {'Entrezgene_track-info': {'Gene-track': {'Gene-track_geneid': '4359',
     'Gene-track_status': {'@value': 'live', '#text': '0'},
     'Gene-track_create-date': {'Date': {'Date_std': {'Date-std': {'Date-std_year': '1998',
         'Date-std_month': '8',
         'Date-std_day': '20'}}}},
     'Gene-track_update-date': {'Date': {'Date_std': {'Date-std': {'Date-std_year': '2026',
         'Date-std_month': '5',
         'Date-std_day': '19',
         'Date-std_hour': '8',
         'Date-std_minute': '32',
         'Date-std_second': '0'}}}}}},
   'Entrezgene_type': {'@value': 'protein-coding', '#text': '6'},
   'Entrezgene_source': {'BioSource': {'BioSource_genome': {'@value': 'genomic',
      '#text': '1'},
     'BioSource_origin': {'@value': 'natural', '#text': '1'},
     'BioSource_org': {'Org-ref': {'Org-ref_taxname': 'Homo sapiens',
       'Org-ref_common': 'human',
       'Org-ref_db': {'Dbtag': {'Dbtag_db': 'taxon',
         'Dbtag_tag': {'Object

In [118]:
record['Entrezgene-Set']['Entrezgene'].keys()

dict_keys(['Entrezgene_track-info', 'Entrezgene_type', 'Entrezgene_source', 'Entrezgene_gene', 'Entrezgene_prot', 'Entrezgene_summary', 'Entrezgene_location', 'Entrezgene_gene-source', 'Entrezgene_locus', 'Entrezgene_properties', 'Entrezgene_comments', 'Entrezgene_unique-keys', 'Entrezgene_xtra-index-terms', 'Entrezgene_xtra-properties'])

In [133]:
record['Entrezgene-Set']['Entrezgene']['Entrezgene_locus']['Gene-commentary'][0]['Gene-commentary_accession']

'NC_000001'

In [134]:
record['Entrezgene-Set']['Entrezgene']['Entrezgene_locus']['Gene-commentary'][0]['Gene-commentary_version']

'11'

In [135]:
get_transcript('NC_000001.11')

{'GBSeq_locus': 'NC_000001', 'GBSeq_length': '248956422', 'GBSeq_strandedness': 'double', 'GBSeq_moltype': 'DNA', 'GBSeq_topology': 'linear', 'GBSeq_division': 'CON', 'GBSeq_update-date': '06-AUG-2025', 'GBSeq_create-date': '29-AUG-2002', 'GBSeq_definition': 'Homo sapiens chromosome 1, GRCh38.p14 Primary Assembly', 'GBSeq_primary-accession': 'NC_000001', 'GBSeq_accession-version': 'NC_000001.11', 'GBSeq_other-seqids': ['gnl|ASM:GCF_000001305|1', 'ref|NC_000001.11|', 'gpp|GPC_000001293.1|', 'gnl|NCBI_GENOMES|1', 'gi|568815597'], 'GBSeq_project': 'PRJNA168', 'GBSeq_keywords': ['RefSeq'], 'GBSeq_source': 'Homo sapiens (human)', 'GBSeq_organism': 'Homo sapiens', 'GBSeq_taxonomy': 'Eukaryota; Metazoa; Chordata; Craniata; Vertebrata; Euteleostomi; Mammalia; Eutheria; Euarchontoglires; Primates; Haplorrhini; Catarrhini; Hominidae; Homo', 'GBSeq_references': [{'GBReference_reference': '1', 'GBReference_position': '1..248956422', 'GBReference_authors': ['Gregory,S.G.', 'Barlow,K.F.', 'McLay,K.E

In [ ]:
for i in record['Entrezgene-Set']['Entrezgene'].keys():
    print(i)
    print()

dict_keys(['Entrezgene_track-info', 'Entrezgene_type', 'Entrezgene_source', 'Entrezgene_gene', 'Entrezgene_prot', 'Entrezgene_summary', 'Entrezgene_location', 'Entrezgene_gene-source', 'Entrezgene_locus', 'Entrezgene_properties', 'Entrezgene_comments', 'Entrezgene_unique-keys', 'Entrezgene_xtra-index-terms', 'Entrezgene_xtra-properties'])

In [113]:
record

{'Entrezgene-Set': {'Entrezgene': {'Entrezgene_track-info': {'Gene-track': {'Gene-track_geneid': '105822916',
     'Gene-track_status': {'@value': 'live', '#text': '0'},
     'Gene-track_create-date': {'Date': {'Date_std': {'Date-std': {'Date-std_year': '2015',
         'Date-std_month': '6',
         'Date-std_day': '1'}}}},
     'Gene-track_update-date': {'Date': {'Date_std': {'Date-std': {'Date-std_year': '2026',
         'Date-std_month': '4',
         'Date-std_day': '8',
         'Date-std_hour': '16',
         'Date-std_minute': '2',
         'Date-std_second': '36'}}}}}},
   'Entrezgene_type': {'@value': 'protein-coding', '#text': '6'},
   'Entrezgene_source': {'BioSource': {'BioSource_genome': {'@value': 'genomic',
      '#text': '1'},
     'BioSource_origin': {'@value': 'natural', '#text': '1'},
     'BioSource_org': {'Org-ref': {'Org-ref_taxname': 'Propithecus coquereli',
       'Org-ref_common': "Coquerel's sifaka",
       'Org-ref_db': {'Dbtag': {'Dbtag_db': 'taxon',
     

In [114]:
orth['CBS']

['875', '736626', '100423684', '100414619', '138387809', '128568008', '126950623', '123630779', '117095839', '116550763', '116482221', '112620305', '111526481', '108529529', '108311214', '105822916', '105713305', '105575589', '104677999', '101007629', '100974696', '100948655', '100597744', '129483030', '129022620', '105879116', '103221104', '102115084', '101149756', '101033999', '100459501', '105472571']

# Reading from ClinVar database

In [145]:
handle = Entrez.efetch(db="clinvar", id='VCV000851448', rettype='vcv', is_varationid=True, from_esearch=True)
tmp = x2d.parse(handle.read().decode('utf-8'))
handle.close()
tmp

{'ClinVarResult-Set': {'set': None}}

In [146]:
get_variant_ids('ABCD1')

['4849478', '4840115', '4822625', '4819835', '4819267', '4819121', '4818089', '4818088', '4809010', '4807940', '4807524', '4806445', '4802208', '4802207', '4802205', '4802204', '4802203', '4802202', '4802201', '4802200', '4802199', '4802198', '4796070', '4795941', '4778430', '4776436', '4774687', '4773396', '4764160', '4756066', '4755890', '4755481', '4754592', '4752726', '4749162', '4744815', '4744352', '4737692', '4736820', '4736681', '4736175', '4735714', '4735020', '4734900', '4732327', '4732184', '4731701', '4731286', '4730515', '4729930', '4723074', '4721692', '4719857', '4719764', '4717975', '4715988', '4715948', '4714903', '4714702', '4713904', '4713550', '4711999', '4710909', '4710908', '4710906', '4710905', '4710903', '4710901', '4710858', '4707858', '4705633', '4705094', '4696104', '4692962', '4689902', '4687895', '4687823', '4686130', '4683026', '4564388', '4564361', '4530964', '4528200', '4526909', '4525879', '4477103', '4477102', '4477101', '4477100', '4477098', '4477097'

In [149]:
rec = get_variant('4849478')
rec

{'@VariationID': '4849478',
 '@VariationName': 'NM_000033.4(ABCD1):c.274G>C (p.Gly92Arg)',
 '@VariationType': 'single nucleotide variant',
 '@Accession': 'VCV004849478',
 '@Version': '1',
 '@RecordType': 'classified',
 '@NumberOfSubmissions': '1',
 '@NumberOfSubmitters': '1',
 '@DateLastUpdated': '2026-05-09',
 '@DateCreated': '2026-05-03',
 '@MostRecentSubmission': '2026-05-03',
 'RecordStatus': 'current',
 'Species': 'Homo sapiens',
 'ClassifiedRecord': {'SimpleAllele': {'@AlleleID': '4960532',
   '@VariationID': '4849478',
   'GeneList': {'Gene': {'@Symbol': 'ABCD1',
     '@FullName': 'ATP binding cassette subfamily D member 1',
     '@GeneID': '215',
     '@HGNC_ID': 'HGNC:61',
     '@Source': 'submitted',
     '@RelationshipType': 'within single gene',
     'Location': {'CytogeneticLocation': 'Xq28',
      'SequenceLocation': [{'@Assembly': 'GRCh38',
        '@AssemblyAccessionVersion': 'GCF_000001405.38',
        '@AssemblyStatus': 'current',
        '@Chr': 'X',
        '@Access

In [153]:
for locus in loci:
    variants = get_variant_ids(locus)
    print(locus, len(variants))

ABCD1 2055
ALPL 1624
AR 1217
ATP7B 3775
BTK 1153
CASR 3126
CBS 1543
CFTR 6387
CYBB 1075
F7 484
F8 1841
F9 888
G6PD 1153
GALT 1079
GBA1 783
GJB1 999
HBB 1958
HPRT1 530
IL2RG 698
KCNH2 3909
KCNQ1 3117
L1CAM 1817
LDLR 4894
MPZ 755
MYH7 5888
TYR 870
PAH 1769
PMM2 985
RHO 675
TP53 3987
TTR 495
VWF 2176
